# BTBC A/B Analysis

Analyzes result JSON files produced by `tests/compare_agents.py`. Memory-level metrics are primary; LLM-answer metrics are secondary. Statistical intervals are exploratory when the number of paired scenarios/runs is small.


In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


## Load one or more result bundles
Point `RESULT_FILES` at baseline and/or ablation/seed result JSONs. The notebook preserves each file's metadata, condition flags, and per-scenario pairing.


In [ ]:
RESULT_FILES = [Path('../results/btbc_memory_ab.json')]

bundles = []
for path in RESULT_FILES:
    obj = json.loads(path.read_text(encoding='utf-8'))
    obj['_path'] = str(path)
    bundles.append(obj)
print(f'Loaded {len(bundles)} bundle(s)')


In [ ]:
def condition_name(bundle):
    ab = bundle.get('ablations', {})
    flags = [k for k, v in ab.items() if v]
    return '+'.join(flags) if flags else 'baseline'

rows = []
for b in bundles:
    meta = b.get('metadata', {})
    for r in b.get('results', []):
        m = r['btbc']['metrics']
        llm = r.get('llm') or {}
        rows.append({
            'file': b['_path'],
            'condition': condition_name(b),
            'seed': meta.get('seed'),
            'repo_sha': meta.get('repo_commit_sha'),
            'model_sha256': meta.get('model_sha256'),
            'scenario_sha256': meta.get('scenario_sha256'),
            'scenario_id': r.get('scenario_id'),
            'plain_error': r['plain'].get('final_memory_error'),
            'btbc_error': m.get('final_memory_error'),
            'error_delta': m.get('final_memory_error') - r['plain'].get('final_memory_error'),
            'true_repairs': m.get('true_repairs', 0),
            'false_corrections': m.get('false_corrections', 0),
            'recovered_corruptions': m.get('recovered_corruptions', 0),
            'recovered_fraction': m.get('recovered_corruption_fraction'),
            'legitimate_change_damage': m.get('legitimate_change_damage', 0),
            'quarantines': m.get('quarantines', 0),
            'escalations': m.get('escalations', 0),
            'repair_decisions': m.get('repair_decisions', 0),
            'plain_answer_exact': (llm.get('plain_score') or {}).get('exact_match'),
            'btbc_answer_exact': (llm.get('btbc_score') or {}).get('exact_match'),
        })
df = pd.DataFrame(rows)
df


## Memory-level summary


In [ ]:
summary = df.groupby('condition', dropna=False).agg(
    scenarios=('scenario_id', 'count'),
    plain_error=('plain_error', 'sum'),
    btbc_error=('btbc_error', 'sum'),
    mean_error_delta=('error_delta', 'mean'),
    true_repairs=('true_repairs', 'sum'),
    false_corrections=('false_corrections', 'sum'),
    recovered_corruptions=('recovered_corruptions', 'sum'),
    legitimate_change_damage=('legitimate_change_damage', 'sum'),
    quarantines=('quarantines', 'sum'),
    escalations=('escalations', 'sum'),
    repair_decisions=('repair_decisions', 'sum'),
).reset_index()
summary['absolute_error_reduction'] = summary['plain_error'] - summary['btbc_error']
summary['relative_error_reduction'] = np.where(summary['plain_error'] > 0, summary['absolute_error_reduction']/summary['plain_error'], np.nan)
summary


## Paired bootstrap confidence interval for memory-error delta
Negative `error_delta = BTBC - plain` favors BTBC. Resampling is paired at the scenario-row level.


In [ ]:
def bootstrap_mean_ci(values, seed=369, n_boot=20000, alpha=0.05):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for i in range(n_boot):
        means[i] = rng.choice(x, size=len(x), replace=True).mean()
    lo, hi = np.quantile(means, [alpha/2, 1-alpha/2])
    return (x.mean(), lo, hi)

ci_rows = []
for cond, g in df.groupby('condition'):
    mean, lo, hi = bootstrap_mean_ci(g['error_delta'])
    ci_rows.append({'condition': cond, 'mean_delta': mean, 'ci95_low': lo, 'ci95_high': hi, 'n': len(g)})
pd.DataFrame(ci_rows)


## Repair / false-correction tradeoff


In [ ]:
trade = summary[['condition','true_repairs','false_corrections','legitimate_change_damage']].copy()
ax = trade.set_index('condition')[['true_repairs','false_corrections','legitimate_change_damage']].plot(kind='bar', figsize=(9,5))
ax.set_ylabel('Count')
ax.set_title('BTBC repair vs damage tradeoff')
plt.tight_layout()
plt.show()


## Per-scenario memory error waterfall


In [ ]:
plot_df = df.sort_values(['condition','error_delta','scenario_id']).copy()
for cond, g in plot_df.groupby('condition'):
    ax = g.set_index('scenario_id')['error_delta'].plot(kind='bar', figsize=(10,4))
    ax.axhline(0, linewidth=1)
    ax.set_ylabel('BTBC error - plain error')
    ax.set_title(f'Per-scenario memory error delta: {cond}')
    plt.tight_layout()
    plt.show()


## LLM paired correctness (when model runs are loaded)
McNemar's exact test here focuses on discordant pairs: plain-correct/BTBC-wrong versus plain-wrong/BTBC-correct.


In [ ]:
llm_df = df.dropna(subset=['plain_answer_exact','btbc_answer_exact']).copy()
if llm_df.empty:
    print('No LLM-scored rows loaded.')
else:
    llm_df['plain_answer_exact'] = llm_df['plain_answer_exact'].astype(bool)
    llm_df['btbc_answer_exact'] = llm_df['btbc_answer_exact'].astype(bool)
    for cond, g in llm_df.groupby('condition'):
        b = int(((g.plain_answer_exact == True) & (g.btbc_answer_exact == False)).sum())
        c = int(((g.plain_answer_exact == False) & (g.btbc_answer_exact == True)).sum())
        discordant = b + c
        p = 1.0 if discordant == 0 else stats.binomtest(min(b,c), discordant, 0.5, alternative='two-sided').pvalue
        print(cond, {'plain_only_correct': b, 'btbc_only_correct': c, 'discordant': discordant, 'mcnemar_exact_p': p})


## Provenance audit
Before treating files as one experiment family, check that frozen hashes/scenario hashes/model hashes are what you intended. Different model hashes should not be pooled as if they were repeated seeds of one model.


In [ ]:
audit = []
for b in bundles:
    m = b.get('metadata', {})
    hashes = m.get('file_hashes', {})
    audit.append({
        'file': b['_path'],
        'condition': condition_name(b),
        'repo_sha': m.get('repo_commit_sha'),
        'seed': m.get('seed'),
        'model_sha256': m.get('model_sha256'),
        'scenario_sha256': m.get('scenario_sha256'),
        'router_sha256': hashes.get('btbc/frozen/router.joblib'),
        'operating_sha256': hashes.get('btbc/frozen/operating.json'),
        'adapter_sha256': hashes.get('btbc/frozen_v1_4_adapter.py'),
        'bridge_sha256': hashes.get('btbc/llm_state_bridge.py'),
    })
pd.DataFrame(audit)


## Interpretation guardrails
- Integration tests establish implementation consistency, not efficacy.
- Memory metrics should be the primary endpoint because they are deterministic and upstream of the LLM.
- LLM correctness should be analyzed as paired outcomes with the same model/prompt/seed.
- Do not pool different scenario hashes, model hashes, or materially different repository commits without labeling them as separate experimental conditions.
- With very small samples, emphasize raw paired outcomes and confidence intervals rather than p-values.
